# Step 10 — Three additions for submission

**A. Seeds.** Pairs (0,1) and (0,8), three seeds each, 80 epochs. Error bars on h_net, fold delay, b, and generated class bias. Also: is the sign of h_net consistently positive?

**B. FashionMNIST.** Same code, different dataset. (i) 45-pair census with cusp fits (CPU). (ii) One trained pair (Trouser=1 vs Sneaker=7): fidelity and bias at ep80.

**C. Label-free entropy-variance detector.** Per-sample conditional-entropy proxy from the denoising error (local Gaussian), `h_i = ½ log(‖D(x_t)−x_0‖²/d)`, and its variance across samples vs τ, computed with the trained (0,1) network. Compare its peak with the exact fold (0.906) and the network's fold. **Check `entropy_proxy` against the Information Dynamics paper's definition before citing it.**

GPU on, Internet on. ~70 min at EPOCHS=80 (set EPOCHS=40 for a ~35 min first pass).

In [ ]:
EPOCHS = 80
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F, math, time, copy
import matplotlib.pyplot as plt
from itertools import combinations
from torchvision import datasets
dev = 'cuda' if torch.cuda.is_available() else 'cpu'; print(dev)
plt.rcParams['figure.dpi'] = 110; SIGMA_DATA = 0.5
def load(ds_cls, root):
    d = ds_cls(root, train=True, download=True); return (d.data.numpy().astype(np.float32)/127.5 - 1.0), d.targets.numpy()
X_MN, Y_MN = load(datasets.MNIST, '/kaggle/working/mnist'); X_FM, Y_FM = load(datasets.FashionMNIST, '/kaggle/working/fmnist')
print(X_MN.shape, X_FM.shape)

## Shared code

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, c, emb):
        super().__init__()
        self.n1 = nn.GroupNorm(8, c); self.c1 = nn.Conv2d(c, c, 3, padding=1)
        self.n2 = nn.GroupNorm(8, c); self.c2 = nn.Conv2d(c, c, 3, padding=1); self.e = nn.Linear(emb, c)
    def forward(self, x, e):
        h = self.c1(F.silu(self.n1(x))) + self.e(e)[:, :, None, None]; return x + self.c2(F.silu(self.n2(h)))
class UNet(nn.Module):
    def __init__(self, ch=64, emb=128):
        super().__init__()
        self.emb = nn.Sequential(nn.Linear(64, emb), nn.SiLU(), nn.Linear(emb, emb)); self.inp = nn.Conv2d(1, ch, 3, padding=1)
        self.d1 = ResBlock(ch, emb); self.down = nn.Conv2d(ch, 2*ch, 3, stride=2, padding=1)
        self.d2 = ResBlock(2*ch, emb); self.mid = ResBlock(2*ch, emb)
        self.up = nn.ConvTranspose2d(2*ch, ch, 4, stride=2, padding=1); self.u1 = ResBlock(ch, emb); self.out = nn.Conv2d(ch, 1, 3, padding=1)
    def forward(self, x, c_noise):
        f = torch.exp(torch.arange(32, device=x.device) * -(math.log(1000)/31))
        e = self.emb(torch.cat([torch.sin(c_noise[:, None]*f), torch.cos(c_noise[:, None]*f)], 1))
        h1 = self.d1(self.inp(x), e); h2 = self.mid(self.d2(self.down(h1), e), e); return self.out(self.u1(self.up(h2) + h1, e))
def denoise(net, x, sigma):
    s = sigma.view(-1, 1, 1, 1); c_skip = SIGMA_DATA**2/(s**2 + SIGMA_DATA**2); c_out = s*SIGMA_DATA/torch.sqrt(s**2 + SIGMA_DATA**2)
    return c_skip*x + c_out*net(x/torch.sqrt(s**2 + SIGMA_DATA**2), 0.25*torch.log(sigma))

def train(X_all, Y_all, pair, seed, epochs=EPOCHS, bs=128, lr=2e-4, P_mean=-0.6, P_std=1.4, ema_decay=0.999, tag=''):
    torch.manual_seed(seed); np.random.seed(seed)
    X = torch.from_numpy(X_all[np.isin(Y_all, pair)])[:, None].to(dev)
    net = UNet().to(dev); ema = copy.deepcopy(net); opt = torch.optim.Adam(net.parameters(), lr=lr); n = len(X); t0 = time.time(); ck = {}
    for ep in range(1, epochs+1):
        perm = torch.randperm(n, device=dev)
        for i in range(0, n-bs+1, bs):
            x0 = X[perm[i:i+bs]]; sigma = torch.exp(P_mean + P_std*torch.randn(bs, device=dev))
            xt = x0 + sigma.view(-1,1,1,1)*torch.randn_like(x0); w = (sigma**2 + SIGMA_DATA**2)/(sigma*SIGMA_DATA)**2
            loss = (w.view(-1,1,1,1)*(denoise(net, xt, sigma) - x0)**2).mean(); opt.zero_grad(); loss.backward(); opt.step()
            with torch.no_grad():
                for pe, pn in zip(ema.parameters(), net.parameters()): pe.lerp_(pn, 1-ema_decay)
        if ep in (20, epochs): ck[ep] = copy.deepcopy(ema).eval(); torch.save(ck[ep].state_dict(), f'/kaggle/working/unet_{tag}{pair[0]}{pair[1]}_s{seed}_ep{ep}.pt')
    print(f"  trained {tag}{pair} seed {seed}: {time.time()-t0:.0f}s"); return ck

rng = np.random.default_rng(0)
GRID = np.linspace(-4, 4, 4001); DG = GRID[1]-GRID[0]; TAU_FIT = np.geomspace(0.3, 2.0, 16); WIN = (TAU_FIT > 0.5) & (TAU_FIT < 1.2)
def smooth_hist(m0, tau):
    h, _ = np.histogram(m0, bins=np.append(GRID-DG/2, GRID[-1]+DG/2), density=True); k = np.exp(-GRID**2/(2*tau)); k /= k.sum()
    return np.maximum(np.convolve(h, k, mode='same'), 1e-300)
def exact_drift(m0, tau):
    p = smooth_hist(m0, tau); return GRID, np.gradient(np.log(p), GRID), p
def _merge(pts, merge=0.02):
    ch = True
    while ch and len(pts) > 1:
        ch = False
        for a in range(len(pts)-1):
            if abs(pts[a][0]-pts[a+1][0]) < merge and pts[a][1] != pts[a+1][1]: del pts[a:a+2]; ch = True; break
    return pts
def stationary_points(p):
    d = np.gradient(np.log(p), GRID); keep = p > 1e-4*p.max()
    idx = np.where((np.sign(d[:-1]) != np.sign(d[1:])) & keep[:-1] & keep[1:])[0]
    return _merge([(float(GRID[i]-d[i]*DG/(d[i+1]-d[i])) if d[i+1] != d[i] else float(GRID[i]), 'max' if d[i] > 0 else 'min') for i in idx])
def n_max(pts): return sum(k == 'max' for _, k in pts)
def birth_tau_bisect(get_pts, taus, iters=12):
    ts = sorted(taus, reverse=True); hi = None
    for tau in ts:
        pts = get_pts(tau)
        if n_max(pts) >= 2 and hi is not None: lo = tau; break
        if n_max(pts) == 1: hi = tau
    else: return None
    for _ in range(iters):
        mid = np.sqrt(lo*hi)
        if n_max(get_pts(mid)) >= 2: lo = mid
        else: hi = mid
    return float(lo)
def lda_direction(Xa, Xb, lam=1.0):
    mu_a, mu_b = Xa.mean(0), Xb.mean(0); delta = mu_a-mu_b
    u = np.linalg.solve(np.cov(Xa.T)+np.cov(Xb.T)+lam*np.eye(784), delta); u /= np.linalg.norm(u)
    return u.astype(np.float32), (0.5*(mu_a+mu_b)).astype(np.float32), 0.5*float(delta @ u)
def balanced_flat(X_all, Y_all, pair):
    Xa = X_all[Y_all == pair[0]].reshape(-1, 784); Xb = X_all[Y_all == pair[1]].reshape(-1, 784)
    nmin = min(len(Xa), len(Xb)); return Xa, Xb, np.concatenate([Xa[:nmin], Xb[:nmin]])
def cusp_fit(m, f, w, window=1.0, b_min=0.02, s_max=0.6):
    ok = (np.abs(m) < window) & (w > 0); c3, c2, c1, c0 = np.polyfit(m[ok], f[ok], 3, w=np.sqrt(w[ok]))
    b = max(-c3, b_min); c3 = -b; s = float(np.clip(-c2/(3*c3), -s_max, s_max))
    a = c1 + 3*c3*s**2 + 2*c2*s; h = c0 + c1*s + c2*s**2 + c3*s**3
    return dict(h=h, a=a, b=b, fold=(a > 0) and (4*a**3 > 27*h**2/b))
@torch.no_grad()
def net_drift(net, Xflat, u, mid, half, tau, n=20000, nbins=70):
    ut = torch.from_numpy(u).to(dev); midt = torch.from_numpy(mid).to(dev); sigma = half*np.sqrt(tau)
    x0 = torch.from_numpy(Xflat[rng.choice(len(Xflat), n)]).to(dev); xt = x0 + sigma*torch.randn_like(x0)
    D = torch.cat([denoise(net, xb.view(-1,1,28,28), torch.full((len(xb),), sigma, device=dev)).view(len(xb), -1) for xb in xt.split(2000)])
    m = ((xt-midt) @ ut / half).cpu().numpy(); sm = (half*((D-xt) @ ut)/sigma**2).cpu().numpy()
    lo, hi = np.quantile(m, [0.005, 0.995]); edges = np.linspace(lo, hi, nbins+1); b = np.clip(np.digitize(m, edges)-1, 0, nbins-1)
    cnt = np.bincount(b, minlength=nbins).astype(float); s = np.bincount(b, weights=sm, minlength=nbins); k = np.exp(-0.5*(np.arange(-5, 6)/1.5)**2)
    return 0.5*(edges[:-1]+edges[1:]), np.convolve(s, k, 'same')/np.maximum(np.convolve(cnt, k, 'same'), 1e-9), cnt
def coeffs(net, Xflat, u, mid, half):
    fits = [cusp_fit(*net_drift(net, Xflat, u, mid, half, t)) for t in TAU_FIT]
    tf = [t for t, f in zip(TAU_FIT, fits) if f['fold']]
    return dict(h=float(np.mean(np.array([f['h'] for f in fits])[WIN])), b=float(np.mean(np.array([f['b'] for f in fits])[WIN])), tau_fold=(max(tf) if tf else np.nan))
def exact_coeffs(m0):
    fits = [cusp_fit(*exact_drift(m0, t)) for t in TAU_FIT]; tf = [t for t, f in zip(TAU_FIT, fits) if f['fold']]
    return dict(h=float(np.mean(np.array([f['h'] for f in fits])[WIN])), b=float(np.mean(np.array([f['b'] for f in fits])[WIN])), tau_fold=(max(tf) if tf else np.nan))
@torch.no_grad()
def gen_fraction(net, u, mid, half, n=2000, steps=40):
    ut = torch.from_numpy(u).to(dev); midt = torch.from_numpy(mid).to(dev)
    i = torch.arange(steps, device=dev); sig = torch.cat([(20**(1/7) + i/(steps-1)*(0.002**(1/7) - 20**(1/7)))**7, torch.zeros(1, device=dev)])
    x = torch.randn(n, 1, 28, 28, device=dev)*sig[0]
    for k in range(steps):
        s0, s1 = sig[k], sig[k+1]; d0 = (x - denoise(net, x, s0.expand(n)))/s0; x1 = x + (s1-s0)*d0
        if s1 > 0: d1 = (x1 - denoise(net, x1, s1.expand(n)))/s1; x = x + (s1-s0)*0.5*(d0+d1)
        else: x = x1
    return float((((x.view(n, -1) - midt) @ ut / half) > 0).float().mean())

## A. Seeds: (0,1) and (0,8) × 3 seeds

In [ ]:
SEEDS = [0, 1, 2]; A = []
for pair in [(0, 1), (0, 8)]:
    Xa, Xb, Xflat = balanced_flat(X_MN, Y_MN, pair); u, mid, half = lda_direction(Xa, Xb); ex = exact_coeffs((Xflat-mid) @ u / half)
    base = float((((Xflat-mid) @ u / half) > 0).mean())
    print(f"\n{pair}: exact h={ex['h']:+.3f} b={ex['b']:.3f} tau_fold={ex['tau_fold']:.3f}; baseline frac={base:.3f}")
    for seed in SEEDS:
        ck = train(X_MN, Y_MN, pair, seed)
        for ep, net in ck.items():
            c = coeffs(net, Xflat, u, mid, half); fr = gen_fraction(net, u, mid, half)
            A.append(dict(pair=pair, seed=seed, ep=ep, **c, excess=fr-base)); print(f"   seed {seed} ep{ep}: h_net={c['h']:+.3f} b={c['b']:.3f} tau_fold={c['tau_fold']:.3f} excess frac={fr-base:+.3f}")
    for ep in (20, EPOCHS):
        r = [x for x in A if x['pair'] == pair and x['ep'] == ep]
        print(f"   ep{ep} mean±sd over seeds: h_net={np.mean([x['h'] for x in r]):+.3f}±{np.std([x['h'] for x in r]):.3f}  "
              f"tau_fold={np.nanmean([x['tau_fold'] for x in r]):.3f}±{np.nanstd([x['tau_fold'] for x in r]):.3f}  "
              f"b={np.mean([x['b'] for x in r]):.3f}±{np.std([x['b'] for x in r]):.3f}  excess={np.mean([x['excess'] for x in r]):+.3f}±{np.std([x['excess'] for x in r]):.3f}  "
              f"sign(h_net)>0 in {sum(x['h'] > 0 for x in r)}/{len(r)}")

## B(i). FashionMNIST census (CPU): cusp fits over 45 balanced pairs

In [ ]:
FM = []
for a_, b_ in combinations(range(10), 2):
    Xa, Xb, Xflat = balanced_flat(X_FM, Y_FM, (a_, b_)); u, mid, half = lda_direction(Xa, Xb); m0 = (Xflat-mid) @ u / half
    tb = birth_tau_bisect(lambda t: stationary_points(smooth_hist(m0, t)), np.geomspace(0.05, 3.0, 60))
    if tb is None: print(f"{(a_,b_)} never bimodal"); continue
    fit = cusp_fit(*exact_drift(m0, tb)); FM.append(dict(pair=(a_, b_), tau=tb, **fit))
H_fm = np.array([abs(f['h']) for f in FM]); H_mn_ref = 0.003
print(f"FashionMNIST: bimodal {len(FM)}/45; |h| median {np.median(H_fm):.3f}, min {H_fm.min():.3f}, max {H_fm.max():.3f}; pairs with |h| <= {H_mn_ref} (pitchfork floor): {int((H_fm <= H_mn_ref).sum())}")
for f in sorted(FM, key=lambda f: abs(f['h'])): print(f"  {f['pair']} tau_bif={f['tau']:.3f} h={f['h']:+.4f} b={f['b']:.3f}")
plt.figure(figsize=(5, 3.4)); plt.hist(H_fm, bins=25); plt.axvline(H_mn_ref, color='k', ls='--', label='pitchfork floor'); plt.xlabel('|h| (FashionMNIST, LDA, balanced)'); plt.legend(); plt.tight_layout(); plt.show()

## B(ii). FashionMNIST trained pair: Trouser (1) vs Sneaker (7)

In [ ]:
pair = (1, 7); Xa, Xb, Xflat = balanced_flat(X_FM, Y_FM, pair); u, mid, half = lda_direction(Xa, Xb); ex = exact_coeffs((Xflat-mid) @ u / half)
base = float((((Xflat-mid) @ u / half) > 0).mean()); ck = train(X_FM, Y_FM, pair, 0, tag='fm')
print(f"FashionMNIST {pair}: exact h={ex['h']:+.3f} b={ex['b']:.3f} tau_fold={ex['tau_fold']:.3f}")
for ep, net in ck.items():
    c = coeffs(net, Xflat, u, mid, half); fr = gen_fraction(net, u, mid, half)
    print(f"   ep{ep}: h_net={c['h']:+.3f} b={c['b']:.3f} tau_fold={c['tau_fold']:.3f} excess frac={fr-base:+.3f}")
FM_NET = ck[EPOCHS]

## C. Label-free entropy-variance detector vs the fold (MNIST (0,1), seed 0, final epoch)

`entropy_proxy` returns a per-sample conditional-entropy estimate. Replace its body with the paper's definition if it differs.

In [ ]:
@torch.no_grad()
def entropy_proxy(net, x0, sigma):
    """Per-sample conditional entropy proxy: local-Gaussian estimate from the denoising error, h_i = 0.5*log(||D(x_t)-x_0||^2/d)."""
    xt = x0 + sigma*torch.randn_like(x0); D = denoise(net, xt.view(-1,1,28,28), torch.full((len(x0),), float(sigma), device=dev)).view(len(x0), -1)
    mse = ((D - x0)**2).mean(1); return 0.5*torch.log(mse + 1e-12), mse

pair = (0, 1); Xa, Xb, Xflat = balanced_flat(X_MN, Y_MN, pair); u, mid, half = lda_direction(Xa, Xb)
net01 = UNet().to(dev); net01.load_state_dict(torch.load(f'/kaggle/working/unet_01_s0_ep{EPOCHS}.pt', map_location=dev)); net01.eval()
taus = np.geomspace(0.05, 5.0, 40); varh, varmse, meanh = [], [], []
x0 = torch.from_numpy(Xflat[rng.choice(len(Xflat), 8000)]).to(dev)
for tau in taus:
    hproxy, mse = entropy_proxy(net01, x0, half*np.sqrt(tau)); varh.append(hproxy.var().item()); varmse.append(mse.var().item()); meanh.append(hproxy.mean().item())
ex_fold = exact_coeffs((Xflat-mid) @ u / half)['tau_fold']; net_fold = coeffs(net01, Xflat, u, mid, half)['tau_fold']
print(f"exact fold tau={ex_fold:.3f}; network fold tau={net_fold:.3f}; Var[h_proxy] peak at tau={taus[int(np.argmax(varh))]:.3f}; Var[mse] peak at tau={taus[int(np.argmax(varmse))]:.3f}")
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
ax[0].plot(taus, varh, 'o-'); ax[0].set_xscale('log'); ax[0].set_title('Var over samples of entropy proxy'); ax[0].axvline(ex_fold, color='k', ls='--', label='exact fold'); ax[0].axvline(net_fold, color='C3', ls='--', label='network fold'); ax[0].legend(); ax[0].set_xlabel('tau')
ax[1].plot(taus, varmse, 'o-'); ax[1].set_xscale('log'); ax[1].set_title('Var over samples of denoising MSE'); ax[1].axvline(ex_fold, color='k', ls='--'); ax[1].axvline(net_fold, color='C3', ls='--'); ax[1].set_xlabel('tau')
plt.tight_layout(); plt.show()

## What to report back

**A.** The per-seed lines and the mean±sd summaries: are h_net, tau_fold, b, and excess consistent across seeds, and is sign(h_net) > 0 in most runs?
**B.** FashionMNIST census summary line and histogram; the (1,7) fidelity/bias lines.
**C.** The detector peak τ vs exact and network fold τ (and confirm whether `entropy_proxy` matches the paper's definition).